# Data Exploration with Pandas

Welcome to your first data analysis notebook! This is a **Jupyter notebook** — an interactive document where you can mix explanations (like this text) with runnable Python code. You execute each code cell by pressing `Shift+Enter`, and the results appear right below.

## What is Pandas?

**Pandas** is Python's most popular library for working with tabular data (think spreadsheets or database tables). It provides a data structure called a **DataFrame** — a table with labeled rows and columns that you can filter, sort, group, and analyze with just a few lines of code.

## What We'll Do

In this notebook, we will:
1. **Load data** from our SQLite database into pandas DataFrames
2. **Explore the data** using `.head()`, `.shape`, `.dtypes`, and `.describe()`
3. **Ask questions** using filtering, `value_counts()`, and `groupby()`
4. **Work with JSON fields** like recipe ingredients
5. **Compute summary statistics** about our food inventory and spending

By the end, you'll have a solid foundation for exploring any dataset with pandas.

## Loading Data from the Database

Our application stores all its data in a **SQLite** database — a lightweight, file-based database that lives at `db/whattoeat.db`. We use **SQLModel** (which builds on SQLAlchemy) to manage the connection.

To get data into pandas, we use `pd.read_sql()`. This function:
1. Takes a SQL query string and a database connection (called an "engine")
2. Sends the query to the database
3. Returns the results as a pandas DataFrame

This is the bridge between the **SQL world** (where data is stored) and the **pandas world** (where we analyze it).

In [ ]:
import sys
from pathlib import Path

# Add the project root to Python's module search path so we can import from src/
# This is needed because Jupyter's working directory is notebooks/, not the project root
project_root = str(Path.cwd().parent) if Path.cwd().name == "notebooks" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
from src.database import get_engine

# Create the database engine — this is our connection to the SQLite file
engine = get_engine()

# pd.read_sql() runs a SQL query and returns the results as a DataFrame
# This is the bridge between SQL (database language) and pandas (Python analysis)
# Each table in our database becomes its own DataFrame
recipes_df = pd.read_sql("SELECT * FROM recipe", engine)
receipts_df = pd.read_sql("SELECT * FROM receipt", engine)
pantry_df = pd.read_sql("SELECT * FROM pantryitem", engine)
inventory_df = pd.read_sql("SELECT * FROM activeinventory", engine)
matching_df = pd.read_sql("SELECT * FROM recipematchsummary", engine)

# Always verify your data loaded correctly by checking row counts
print(f"Loaded {len(recipes_df)} recipes, {len(receipts_df)} receipts, "
      f"{len(pantry_df)} pantry items, {len(inventory_df)} inventory items, "
      f"{len(matching_df)} recipe match summaries")

## First Look at the Data

When you get a new dataset, the first thing to do is **look at it**. Pandas gives you several tools for a quick overview:

| Method | What it does |
|--------|-------------|
| `.head()` | Shows the first 5 rows (like peeking at the top of a spreadsheet) |
| `.shape` | Returns `(rows, columns)` — tells you how big the data is |
| `.dtypes` | Shows the data type of each column (text, numbers, dates, etc.) |
| `.describe()` | Computes summary statistics (mean, min, max, etc.) for numeric columns |

Let's try each one.

In [ ]:
# .head() shows the first 5 rows — a quick preview of the data
# This is always the first thing you should do with a new DataFrame
# Tip: you can pass a number like .head(10) to see more rows
recipes_df.head()

In [ ]:
# .shape returns (rows, columns) — tells you the size of your data
# This is useful for understanding how much data you're working with
print(f"Recipes table: {recipes_df.shape[0]} rows, {recipes_df.shape[1]} columns")
print(f"Receipts table: {receipts_df.shape[0]} rows, {receipts_df.shape[1]} columns")
print(f"Pantry table: {pantry_df.shape[0]} rows, {pantry_df.shape[1]} columns")
print(f"Inventory table: {inventory_df.shape[0]} rows, {inventory_df.shape[1]} columns")

# .dtypes shows the data type of each column — important for knowing
# what operations you can perform (e.g., math only works on numbers)
# Common types: object (text/JSON), int64 (integers), float64 (decimals)
print("\nRecipe column types:")
print(recipes_df.dtypes)

In [ ]:
# .describe() computes summary statistics for numeric columns:
# count, mean, std, min, 25%, 50% (median), 75%, max
# This is a fast way to spot data quality issues (negative prices? zero quantities?)
# If you see unexpected values here, it often means there's a data problem to investigate
receipts_df.describe()

## Asking Questions of Your Data

Now that we know what the data looks like, let's start asking questions. Pandas has three key tools for this:

- **`value_counts()`** — "How many of each?" Count occurrences of each unique value in a column
- **Boolean filtering** — "Which rows match this condition?" Select rows that meet specific criteria
- **`groupby()`** — "For each group, compute something." Split data into groups and aggregate

These three operations will handle the vast majority of your data questions.

In [ ]:
# value_counts() answers "how many of each?" — the most common first
# It's like a frequency table or histogram in number form
# This is great for understanding the distribution of categorical data

print("Recipes by weather temperature:")
print(recipes_df["weather_temp"].value_counts())

print("\nRecipes by weather condition:")
print(recipes_df["weather_condition"].value_counts())

In [ ]:
# Boolean filtering: df[condition] keeps only rows where condition is True
# Think of it as asking a yes/no question about every row

# Find all cold-weather recipes
cold_recipes = recipes_df[recipes_df["weather_temp"] == "cold"]
print(f"Cold weather recipes ({len(cold_recipes)}):")
for _, recipe in cold_recipes.iterrows():
    print(f"  - {recipe['name']}")

# You can combine conditions with & (AND) and | (OR)
# Important: each condition must be in parentheses!
cold_rainy = recipes_df[
    (recipes_df["weather_temp"] == "cold") & 
    (recipes_df["weather_condition"] == "rainy")
]
print(f"\nCold AND rainy recipes ({len(cold_rainy)}):")
for _, recipe in cold_rainy.iterrows():
    print(f"  - {recipe['name']}")

In [ ]:
# groupby() splits data into groups, then applies a function to each group
# Like: "for each category, count how many items there are"
# This is one of the most powerful operations in pandas

print("Inventory items by category:")
category_counts = inventory_df.groupby("category").size()
# .sort_values(ascending=False) puts the largest groups first
print(category_counts.sort_values(ascending=False))

## Working with JSON Fields

Some columns in our database store **JSON** (JavaScript Object Notation) — a format for nested, structured data. For example, each recipe's `ingredients` column contains a JSON list of dictionaries, where each dictionary has keys like `name`, `quantity`, `unit`, and `category`.

In SQLite, JSON is stored as plain text. To work with it in pandas, we need to **parse** it back into Python objects using `json.loads()`. Once parsed, we can extract, count, and analyze the nested data.

In [ ]:
import json

# The 'ingredients' column stores JSON — a list of dictionaries.
# In SQLite, JSON is stored as text. We need to parse it back to Python objects.

# json.loads() converts a JSON string back to a Python list/dict
# We use .apply() to run this function on every row
recipes_df["parsed_ingredients"] = recipes_df["ingredients"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

# Now we can work with the parsed data — e.g., count ingredients per recipe
recipes_df["ingredient_count"] = recipes_df["parsed_ingredients"].apply(len)

# Display recipe names alongside their ingredient counts, sorted by most ingredients
print("Ingredients per recipe:")
print(recipes_df[["name", "ingredient_count"]].sort_values(
    "ingredient_count", ascending=False
).to_string(index=False))

In [ ]:
# explode() "unnests" a list column — if a recipe has 8 ingredients,
# it creates 8 rows (one per ingredient), each still linked to the recipe.
# This lets us analyze individual ingredients across ALL recipes.

# Step 1: Extract just the ingredient names from each recipe
ingredient_names = recipes_df["parsed_ingredients"].apply(
    lambda ings: [ing["name"] for ing in ings if isinstance(ing, dict)]
)

# Step 2: Explode to one row per ingredient
# Before: one row = one recipe with a list of ingredient names
# After:  one row = one ingredient (recipe info is repeated for each)
all_ingredients = ingredient_names.explode()

# Step 3: Count how many recipes each ingredient appears in
# This tells us which ingredients are most versatile / commonly used
print("Top 15 most common ingredients across all recipes:")
print(all_ingredients.value_counts().head(15))

## Summary Statistics

Let's bring it all together with some summary statistics about our food data. The `.agg()` method lets you apply **multiple functions at once** to a column — perfect for getting a complete picture in one operation.

We'll also combine several of the techniques we've learned (filtering, counting, `nunique()`) to build a comprehensive summary of our inventory and spending.

In [ ]:
# .agg() lets you apply MULTIPLE functions at once
# This is powerful for getting a complete picture in one operation

if "total_price" in receipts_df.columns:
    receipt_stats = receipts_df["total_price"].agg(["sum", "mean", "min", "max", "count"])
    print("Receipt price statistics:")
    print(f"  Total spent:    ${receipt_stats['sum']:.2f}")
    print(f"  Average price:  ${receipt_stats['mean']:.2f}")
    print(f"  Cheapest item:  ${receipt_stats['min']:.2f}")
    print(f"  Most expensive: ${receipt_stats['max']:.2f}")
    print(f"  Total items:    {int(receipt_stats['count'])}")

# Inventory statistics using filtering and nunique()
# nunique() counts the number of distinct values — like COUNT(DISTINCT ...) in SQL
print("\nInventory summary:")
print(f"  Total items: {len(inventory_df)}")
print(f"  Active (not expired): {len(inventory_df[inventory_df['is_expired'] == False])}")
print(f"  Expired: {len(inventory_df[inventory_df['is_expired'] == True])}")
print(f"  Categories: {inventory_df['category'].nunique()}")
print(f"  Data sources: {inventory_df['source'].value_counts().to_dict()}")

## Exercises to Try

Now it's your turn! Add new code cells below and try these challenges:

- **Exercise 1:** Modify the `groupby` example above to find the **average number of ingredients** per `weather_temp` category. *Hint: use `recipes_df.groupby("weather_temp")["ingredient_count"].mean()`*

- **Exercise 2:** Filter to only recipes with **more than 8 ingredients**. What do they have in common? Look at their weather categories, cook times, and names for patterns.

- **Exercise 3:** Find which **category of food** appears in the most recipes. *Hint: parse the ingredients JSON, extract the `"category"` field from each ingredient dict, explode, and use `value_counts()`.*

Remember: in Jupyter, you can always add a new cell with the `+` button or by pressing `B` (for "below") in command mode. Experiment freely — you can't break anything!